In [1]:
import matplotlib.pyplot as pyplot
import pandas as pd
import numpy as np
import requests
from google.colab import drive

In [2]:
drive.mount("/content/drive/")

Mounted at /content/drive/


In [3]:
storm_events = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/storm_events.parquet")
precip_soil_nino_density_poverty = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/precip_soil_nino_density_poverty.parquet")
static_all = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/updated_static_all.parquet")
temp_wind = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/temp_wind_dynamic.parquet")

In [4]:
storm_events.drop(columns=["DAMAGE_PROPERTY", "DEATHS_TOTAL"], inplace=True)

In [5]:
precip_soil_nino_density_poverty[precip_soil_nino_density_poverty["year"] == 2024]
storm_events_precip_soil_nino_density_poverty = storm_events.merge(precip_soil_nino_density_poverty, on=["GEOID", "date"], how="left")

In [6]:
# backfill nantucket's nan values with nearby barnstable
temp_wind = temp_wind[temp_wind["GEOID"] != "25019"]
proxy = temp_wind[temp_wind["GEOID"] == "25001"][["date","wind_speed","temperature_2m_anomaly","dewpoint_temperature_2m_anomaly"]].copy()
proxy["GEOID"] = "25019"
temp_wind = pd.concat([temp_wind, proxy], ignore_index=True)

storm_events_precip_soil_nino_density_poverty_temp_wind = storm_events_precip_soil_nino_density_poverty.merge(temp_wind, on=["GEOID", "date"], how="left")
storm_events_precip_soil_nino_density_poverty_temp_wind

,GEOID,date,damage_label_t+3,damage_label,death_label_t+3,death_label,damage_antecedent_60,damage_antecedent_12,deaths_antecedent_60,deaths_antecedent_12,...,precip_prior6,nino_3.4,year,population,area_sqkm,pop_density,poverty_rate,wind_speed,temperature_2m_anomaly,dewpoint_temperature_2m_anomaly
0,01001,1996-01,55401.281071,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,...,0.608666,-0.86,1996,40207.0,1539.602123,26.115189,11.8,0.557047,-1.049767,-0.525004
1,01001,1996-02,20518.992989,73868.374761,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.706057,-0.86,1996,40207.0,1539.602123,26.115189,11.8,0.904787,-1.263824,-2.199961
2,01001,1996-03,0.000000,55401.281071,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.732588,-0.61,1996,40207.0,1539.602123,26.115189,11.8,0.381523,-3.384099,-3.079653
3,01001,1996-04,0.000000,20518.992989,0.0,0.0,24622.791587,24622.791587,0.0,0.0,...,0.968626,-0.47,1996,40207.0,1539.602123,26.115189,11.8,0.871127,-2.341284,-1.686003
4,01001,1996-05,20518.992989,0.000000,0.0,0.0,32317.413958,32317.413958,0.0,0.0,...,0.835682,-0.38,1996,40207.0,1539.602123,26.115189,11.8,0.894498,1.025688,0.753520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1128199,56045,2025-11,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.319473,-0.70,2025,NaN,NaN,NaN,NaN,1.184760,2.939855,3.547939
1128200,56045,2025-12,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.262513,-0.67,2025,NaN,NaN,NaN,NaN,2.990652,4.803338,3.548774
1128201,56045,2026-01,NaN,0.000000,NaN,0.0,0.000000,0.000000,0.0,0.0,...,0.192637,-0.58,2026,NaN,NaN,NaN,NaN,3.987307,3.366800,-0.080693
1128202,56045,2026-02,NaN,0.000000,NaN,0.0,0.000000,0.000000,0.0,0.0,...,0.141030,-0.27,2026,NaN,NaN,NaN,NaN,1.921234,6.154798,1.917538


In [7]:
static_all = static_all.drop_duplicates(subset="GEOID")
complete_df = storm_events_precip_soil_nino_density_poverty_temp_wind.merge(static_all, on="GEOID", how="left")
complete_df

,GEOID,date,damage_label_t+3,damage_label,death_label_t+3,death_label,damage_antecedent_60,damage_antecedent_12,deaths_antecedent_60,deaths_antecedent_12,...,pop_density,poverty_rate,wind_speed,temperature_2m_anomaly,dewpoint_temperature_2m_anomaly,hand_vulnerability,dist_coast_m,pct_no_vehicle,lat,lon
0,01001,1996-01,55401.281071,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,...,26.115189,11.8,0.557047,-1.049767,-0.525004,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
1,01001,1996-02,20518.992989,73868.374761,0.0,0.0,0.000000,0.000000,0.0,0.0,...,26.115189,11.8,0.904787,-1.263824,-2.199961,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
2,01001,1996-03,0.000000,55401.281071,0.0,0.0,0.000000,0.000000,0.0,0.0,...,26.115189,11.8,0.381523,-3.384099,-3.079653,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
3,01001,1996-04,0.000000,20518.992989,0.0,0.0,24622.791587,24622.791587,0.0,0.0,...,26.115189,11.8,0.871127,-2.341284,-1.686003,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
4,01001,1996-05,20518.992989,0.000000,0.0,0.0,32317.413958,32317.413958,0.0,0.0,...,26.115189,11.8,0.894498,1.025688,0.753520,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1128199,56045,2025-11,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,NaN,NaN,1.184760,2.939855,3.547939,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128200,56045,2025-12,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,NaN,NaN,2.990652,4.803338,3.548774,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128201,56045,2026-01,NaN,0.000000,NaN,0.0,0.000000,0.000000,0.0,0.0,...,NaN,NaN,3.987307,3.366800,-0.080693,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128202,56045,2026-02,NaN,0.000000,NaN,0.0,0.000000,0.000000,0.0,0.0,...,NaN,NaN,1.921234,6.154798,1.917538,0.258587,1.387298e+06,0.026453,43.840147,-104.567843


In [8]:
print(complete_df.isna().sum())

GEOID                                  0
date                                   0
damage_label_t+3                    9324
damage_label                        3108
death_label_t+3                     9324
death_label                         3108
damage_antecedent_60                3108
damage_antecedent_12                3108
deaths_antecedent_60                3108
deaths_antecedent_12                3108
volumetric_soil_water_layer_3          0
precip_prior3                          0
precip_prior6                          0
nino_3.4                               0
year                                   0
population                         47328
area_sqkm                          47328
pop_density                        47328
poverty_rate                       47208
wind_speed                             0
temperature_2m_anomaly                 0
dewpoint_temperature_2m_anomaly        0
hand_vulnerability                     0
dist_coast_m                           0
pct_no_vehicle  

In [9]:
# certain counties lack pop values and poverty rate values; this can be solved by backfilling those missing in the early years and forward filling
# the gap after 2024
print(complete_df[complete_df["GEOID"]=="51019"]["population"].isna().sum())

miss = complete_df[complete_df["population"].isna()]
print(miss["date"].str[:4].value_counts().sort_index())   # which years
print(miss["GEOID"].nunique(), "counties affected")
print(miss.groupby("GEOID").size().describe())            # even across counties?

complete_df = complete_df.sort_values(["GEOID", "date"])
for col in ["population", "area_sqkm", "pop_density", "poverty_rate"]:
  complete_df[col] = complete_df.groupby("GEOID")[col].ffill()
  complete_df[col] = complete_df.groupby("GEOID")[col].bfill()

print("\n", complete_df[["population","area_sqkm","pop_density","poverty_rate"]].isna().sum())
complete_df = complete_df.drop(columns="area_sqkm")


# BEDFORD STILL NAN
print(complete_df[complete_df["GEOID"]=="51019"]["population"].isna().sum())

363
date
1996       72
1997       72
1998       72
1999       72
2000       72
2001       72
2002       12
2003       12
2004       12
2005       12
2006       12
2007       12
2008       12
2009       12
2010       12
2011       12
2012       12
2013       12
2014       12
2015       12
2016       12
2017       12
2018       12
2019       12
2020       12
2021       12
2022       12
2023       12
2024       12
2025    37296
2026     9324
Name: count, dtype: int64
3108 counties affected
count    3108.000000
mean       15.227799
std         6.875197
min        15.000000
25%        15.000000
50%        15.000000
75%        15.000000
max       363.000000
dtype: float64

 population      363
area_sqkm       363
pop_density     363
poverty_rate      0
dtype: int64
363


In [10]:
persistent = complete_df[(complete_df["population"].isna()) &
                         (complete_df["date"].str[:4].between("2002","2024"))]
print(persistent["GEOID"].unique())

# bedford's geoid

['51019']


In [11]:
b = complete_df[complete_df["GEOID"]=="51019"]["population"]
print("non-NaN count:", b.isna().sum())
print("unique non-NaN values:", b.dropna().unique())

non-NaN count: 363
unique non-NaN values: []


In [12]:
# FRED series for Bedford County VA resident population
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VABEDF9POP"
bedford = pd.read_csv(url)
bedford.columns = ["date", "population"]
bedford["year"] = pd.to_datetime(bedford["date"]).dt.year
bedford["population"] = bedford["population"] * 1000
print(bedford[bedford["year"].between(1996, 2025)])

# map bedford's pop values onto complete_df
bedford_pop = dict(zip(bedford["year"], bedford["population"]))
mask = complete_df["GEOID"] == "51019"
complete_df.loc[mask, "population"] = complete_df.loc[mask, "date"].str[0:4].astype(int).map(bedford_pop)

# extend to 2026
complete_df.loc[mask, "population"] = complete_df.loc[mask,"population"].ffill()

# create the population density by dividing by 1990 sq km (bedford's area)
complete_df.loc[mask, "pop_density"] = complete_df.loc[mask, "population"] / 1990

print(complete_df[mask][["date","population","pop_density"]].iloc[::24])

          date  population  year
26  1996-01-01     56308.0  1996
27  1997-01-01     57031.0  1997
28  1998-01-01     58477.0  1998
29  1999-01-01     59436.0  1999
30  2000-01-01     60457.0  2000
31  2001-01-01     60840.0  2001
32  2002-01-01     61223.0  2002
33  2003-01-01     61948.0  2003
34  2004-01-01     62912.0  2004
35  2005-01-01     64011.0  2005
36  2006-01-01     65386.0  2006
37  2007-01-01     66305.0  2007
38  2008-01-01     66400.0  2008
39  2009-01-01     67081.0  2009
40  2010-01-01     75032.0  2010
41  2011-01-01     75555.0  2011
42  2012-01-01     75755.0  2012
43  2013-01-01     76189.0  2013
44  2014-01-01     77054.0  2014
45  2015-01-01     77653.0  2015
46  2016-01-01     78024.0  2016
47  2017-01-01     78445.0  2017
48  2018-01-01     79115.0  2018
49  2019-01-01     79431.0  2019
50  2020-01-01     79590.0  2020
51  2021-01-01     80311.0  2021
52  2022-01-01     80832.0  2022
53  2023-01-01     81765.0  2023
54  2024-01-01     82467.0  2024
55  2025-0

In [13]:
# set the first month of nan values for deaths/damage_antecdent_60/12 to 0
ante = ["damage_antecedent_60","damage_antecedent_12","deaths_antecedent_60","deaths_antecedent_12"]
complete_df[ante] = complete_df[ante].fillna(0.0)

# and remove the last 5 months for each county which are na because the labels need to sum t+3, t+4, t+5
complete_df = complete_df.dropna(subset=["damage_label", "death_label"])

In [14]:
complete_df.isna().sum()

,0
GEOID,0
date,0
damage_label_t+3,6216
damage_label,0
death_label_t+3,6216
death_label,0
damage_antecedent_60,0
damage_antecedent_12,0
deaths_antecedent_60,0
deaths_antecedent_12,0


In [15]:
complete_df["year"] = complete_df["date"].str[0:4].astype(int)
complete_df.columns

Index(['GEOID', 'date', 'damage_label_t+3', 'damage_label', 'death_label_t+3',
       'death_label', 'damage_antecedent_60', 'damage_antecedent_12',
       'deaths_antecedent_60', 'deaths_antecedent_12',
       'volumetric_soil_water_layer_3', 'precip_prior3', 'precip_prior6',
       'nino_3.4', 'year', 'population', 'pop_density', 'poverty_rate',
       'wind_speed', 'temperature_2m_anomaly',
       'dewpoint_temperature_2m_anomaly', 'hand_vulnerability', 'dist_coast_m',
       'pct_no_vehicle', 'lat', 'lon'],
      dtype='object')

In [16]:
complete_df.to_parquet("/content/drive/MyDrive/Nino/finished_data/complete_df.parquet")
complete_df

,GEOID,date,damage_label_t+3,damage_label,death_label_t+3,death_label,damage_antecedent_60,damage_antecedent_12,deaths_antecedent_60,deaths_antecedent_12,...,pop_density,poverty_rate,wind_speed,temperature_2m_anomaly,dewpoint_temperature_2m_anomaly,hand_vulnerability,dist_coast_m,pct_no_vehicle,lat,lon
0,01001,1996-01,55401.281071,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,26.115189,11.8,0.557047,-1.049767,-0.525004,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
1,01001,1996-02,20518.992989,73868.374761,0.0,0.0,0.000000,0.000000,0.0,0.0,...,26.115189,11.8,0.904787,-1.263824,-2.199961,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
2,01001,1996-03,0.000000,55401.281071,0.0,0.0,0.000000,0.000000,0.0,0.0,...,26.115189,11.8,0.381523,-3.384099,-3.079653,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
3,01001,1996-04,0.000000,20518.992989,0.0,0.0,24622.791587,24622.791587,0.0,0.0,...,26.115189,11.8,0.871127,-2.341284,-1.686003,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
4,01001,1996-05,20518.992989,0.000000,0.0,0.0,32317.413958,32317.413958,0.0,0.0,...,26.115189,11.8,0.894498,1.025688,0.753520,0.275304,2.154338e+05,0.056405,32.534912,-86.642761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1128198,56045,2025-10,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,1.105493,10.5,0.670499,1.852868,2.261861,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128199,56045,2025-11,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,1.105493,10.5,1.184760,2.939855,3.547939,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128200,56045,2025-12,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,...,1.105493,10.5,2.990652,4.803338,3.548774,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
1128201,56045,2026-01,NaN,0.000000,NaN,0.0,0.000000,0.000000,0.0,0.0,...,1.105493,10.5,3.987307,3.366800,-0.080693,0.258587,1.387298e+06,0.026453,43.840147,-104.567843


In [23]:
show = complete_df[(complete_df["GEOID"] == "56045") & (complete_df["date"] == "2026-02")]
with pd.option_context('display.max_columns', None):
    display(show)

,GEOID,date,damage_label_t+3,damage_label,death_label_t+3,death_label,damage_antecedent_60,damage_antecedent_12,deaths_antecedent_60,deaths_antecedent_12,volumetric_soil_water_layer_3,precip_prior3,precip_prior6,nino_3.4,year,population,pop_density,poverty_rate,wind_speed,temperature_2m_anomaly,dewpoint_temperature_2m_anomaly,hand_vulnerability,dist_coast_m,pct_no_vehicle,lat,lon
1128202,56045,2026-02,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.192497,0.04431,0.14103,-0.27,2026,6866.0,1.105493,10.5,1.921234,6.154798,1.917538,0.258587,1.387298e+06,0.026453,43.840147,-104.567843
